# 04 - Modeling: Pelatihan LDA Final (K=8)## TujuanMelatih model Latent Dirichlet Allocation dengan konfigurasi optimal danmenyimpan seluruh artefak model.## Asal (Input)`data/intermediate/` - korpus bersih 285 dokumen hasil Notebook 03## Keluaran (Output)- `model/lda_model.gensim` - model LDA biner (K=8)- `model/topic_distribution.csv` - topik dominan per dokumen- `model/topic_labels.csv` / `.json` - label 8 topik- `model/evaluation_metrics.csv` - metrik model- `model/human_topic_validation.csv` - lembar validasiexpert- `model/lda_visualization.html` - visualisasi pyLDAvis## Konfigurasi Final| Parameter | Nilai ||:---|:---|| Jumlah topik (K) | 8 || Alpha | asymmetric || Eta | auto || Passes | 30 || Iterations | 500 || Random seed | 42 || Metode inferensi | Online Variational Bayes (bukan Gibbs Sampling) |

In [ ]:
import sys, io as _iofrom pathlib import Path_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(_ROOT))try:    sys.stdout.reconfigure(encoding='utf-8', errors='replace')except Exception:    passimport pandas as pd, numpy as npfrom src import config as CC.ensure_dirs()print(C.describe())

## Catatan Metodologis Penting`modeling.py` melakukan `train_test_split(80/20)`: model dilatih pada228 dokumen, lalu diprediksi ke seluruh 285. Dokumen di hold-out (20%)belum pernah dilihat model sehingga theta-nya diffuse.Akibatnya, distribusi di `model/topic_distribution.csv` bisa berbeda 1-2dokumen dari distribusi final. **Distribusi final yang dipakai dashboarddan skripsi dihitung dengan inferensi penuh** - lihat Notebook 08.

## Langkah 1 - Validasi Fingerprint Label

In [ ]:
from src.topic_labels import validate_labelsvalidate_labels(str(C.LDA_MODEL), str(C.LABELS_JSON))print('[OK] Fingerprint label cocok dengan model')

## Langkah 2 - Latih Model LDA FinalSel berikut melatih ulang model LDA dengan konfigurasi final.Butuh sekitar 5 menit (30 passes x 500 iterasi).Untuk sekadar **memverifikasi** model yang sudah ada, lewati sel ini danlangsung ke Langkah 3. Set `RETRAIN = False` agar tidakMexecute.

In [ ]:
RETRAIN = False   # set True untuk melatih ulang dari nolif RETRAIN:    from src import modeling    modeling.main()else:    print('[SKIP] Melewati pelatihan ulang - menggunakan model yang tersimpan')    print('       (set RETRAIN = True jika ingin dilatih dari nol)')

## Langkah 3 - Verifikasi Model Tersimpan

In [ ]:
from gensim.models import LdaModelfrom gensim import corporam = LdaModel.load(str(C.LDA_MODEL))d = corpora.Dictionary.load(str(C.DICTIONARY_PATH))print(f'K              : {m.num_topics}')print(f'Vocabulary     : {len(d)}')print(f'Passes         : {m.passes}')print(f'Iterations     : {m.iterations}')print(f'Alpha (asymm.) : {m.alpha[:3]}...')print(f'Eta (auto)     : {m.eta[:3]}...')

## Langkah 4 - Hitung Coherence Score C_V (Protokol Resmi)

In [ ]:
import astfrom gensim.models import CoherenceModelprep = pd.read_csv(C.PREPROCESSED_CSV)texts = prep['tokens'].apply(ast.literal_eval).tolist()cv = CoherenceModel(model=m, texts=texts, dictionary=d, coherence='c_v', processes=1).get_coherence()print(f'C_V terhitung : {cv:.4f}')print(f'C_V resmi     : {C.COHERENCE_CV_OFFICIAL}')print('Status        :', 'SESHUI' if abs(cv - C.COHERENCE_CV_OFFICIAL) < 0.0001 else 'BERBEDA')

## Langkah 5 - Kata Kunci Setiap Topik

In [ ]:
for t in range(m.num_topics):    words = [w for w, _ in m.show_topic(t, topn=8)]    print(f'  Topik {t+1}: ' + ', '.join(words))

## Ringkasan Fase 04 - Modeling| Metrik | Nilai ||:---|---:|| Jumlah topik (K) | 8 || Coherence C_V | **0,4256** || Vocabulary | 1.849 |Model final siap digunakan pada tahap Evaluation (Notebook 05) danBuild Final Artifacts (Notebook 08).

## Peta Artefak| Artefak | Dibuat oleh | Consumed by ||:---|:---|:---|| `model/lda_model.gensim` | Notebook 04 | Notebook 05, 08, app || `model/topic_distribution.csv` | Notebook 04 | Referensi perbandingan || `model/topic_labels.csv` / `.json` | Notebook 04 | Notebook 07, app || `model/evaluation_metrics.csv` | Notebook 04 | Dokumentasi |